# 🔍 Salient Object Detection — Demo Notebook
Run each cell in order. Upload any image in the last cell to see the model's prediction.

In [ ]:
# ── Install dependencies (Colab / Kaggle) ──────────────────
# !pip install torch torchvision tqdm matplotlib pillow gradio -q

In [ ]:
import sys, os, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image

import torch
import torchvision.transforms.functional as TF

# Add project root to path if running from notebook
sys.path.insert(0, str(Path('.').resolve()))

from sod_model import SODNet

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE  = 224
THRESHOLD   = 0.5
MODEL_PATH  = 'outputs/best_model.pth'   # ← change if needed

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_PATH}')

In [ ]:
# ── Load model ─────────────────────────────────────────────
model = SODNet(base_filters=32).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f'Loaded {model.count_parameters():,} parameters')

In [ ]:
# ── Inference helper ───────────────────────────────────────
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def run_inference(image_path: str, threshold: float = THRESHOLD):
    pil = Image.open(image_path).convert('RGB')
    orig_size = pil.size   # (W, H)

    img = pil.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    t   = TF.to_tensor(img)
    t   = TF.normalize(t, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    t   = t.unsqueeze(0).to(DEVICE)

    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(t)
    ms = (time.perf_counter() - t0) * 1000

    pred_np  = pred.squeeze().cpu().numpy()
    mask_bin = (pred_np > threshold).astype(np.uint8) * 255
    mask_pil = Image.fromarray(mask_bin, 'L').resize(orig_size, Image.NEAREST)

    # Overlay
    orig_np = np.array(pil, dtype=np.float32)
    m_np    = np.array(mask_pil, dtype=np.float32) / 255.0
    ov      = orig_np.copy()
    ov[..., 1] = np.clip(ov[..., 1] * 0.6 + m_np * 200, 0, 255)
    overlay_pil = Image.fromarray(ov.astype(np.uint8))

    return pil, mask_pil, overlay_pil, ms


def show_result(image_path: str, threshold: float = THRESHOLD):
    pil, mask, overlay, ms = run_inference(image_path, threshold)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ['Input Image', 'Saliency Mask', 'Overlay']
    imgs   = [pil, mask, overlay]

    for ax, title, img in zip(axes, titles, imgs):
        ax.imshow(img, cmap='gray' if title == 'Saliency Mask' else None)
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.suptitle(f'Inference time: {ms:.1f} ms', y=1.01, fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'⏱ Inference time: {ms:.1f} ms')

print('Helper functions ready.')

In [ ]:
# ── Option A: Use a local file path ────────────────────────
# show_result('my_image.jpg')


# ── Option B: Upload via file dialog (Jupyter / Colab) ─────
from ipywidgets import FileUpload, Output
import ipywidgets as widgets
from io import BytesIO

uploader = FileUpload(accept='image/*', multiple=False)
out      = Output()

def on_upload(change):
    out.clear_output()
    with out:
        content = list(uploader.value.values())[0]['content']
        pil     = Image.open(BytesIO(content)).convert('RGB')
        # Save temp
        pil.save('/tmp/uploaded.jpg')
        show_result('/tmp/uploaded.jpg')

uploader.observe(on_upload, names='value')
display(uploader, out)

In [ ]:
# ── Plot training curves (if log exists) ───────────────────
log_path = 'outputs/training_log.csv'

if Path(log_path).exists():
    from evaluate import plot_training_curves
    plot_training_curves(log_path, save_path='outputs/training_curves.png')
    display(Image.open('outputs/training_curves.png'))
else:
    print('No training log found at', log_path)

In [ ]:
# ── Run full evaluation on test set ────────────────────────
# from evaluate import evaluate
#
# results = evaluate(
#     model_path   = MODEL_PATH,
#     image_dir    = 'dataset/images',
#     mask_dir     = 'dataset/masks',
#     output_dir   = 'outputs',
#     image_size   = IMAGE_SIZE,
#     batch_size   = 8,
# )
# print(results)